# Aislar por qué el promedio simple con PEFT salió mal — Google Colab

El promedio simple con PEFT (pesos 1.0, método `linear`) dio 35.3 BLEU, por debajo del modelo base, y el de `mergekit` 44.2. Eran **dos diferencias a la vez** (ver `merging/analisis_lineal_peft.md`): (1) PEFT con pesos 1.0 *suma* las tres actualizaciones en vez de promediarlas, y (2) PEFT promedia por separado A y B, lo que mete 6 términos cruzados `B_i A_j` (ruido: las matrices A de adaptadores distintos son ortogonales).

Esta libreta evalúa dos variantes que las separan:

| Variante | Escala correcta (promedio) | Sin términos cruzados |
|---|---|---|
| `peft_linear_norm` (PEFT `linear`, pesos 1/3) | sí | **no** |
| `peft_cat_norm` (PEFT `cat`, pesos 1/3, rank 24) | sí | **sí** (promedio exacto) |

**Cómo leer el resultado:** si `peft_linear_norm` ya funciona (~44), la causa principal era la escala. Si falla y `peft_cat_norm` funciona, la causa principal eran los términos cruzados. Si ambas fallan, hay otra causa. Tarda ~20 min en T4 (2 × ~5.6 min de predicciones). **Entorno de ejecución → T4 GPU.** Si algo falla, pásame el error tal cual.

In [ ]:
import torch
!nvidia-smi
assert torch.cuda.is_available(), "SIN GPU: cambia el tipo de entorno a T4 GPU"

In [ ]:
import os, json, time
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop
!pip install -q transformers peft accelerate sacrebleu python-dotenv
!pip uninstall -y -q torchao
os.makedirs("merging/logs", exist_ok=True)
TIEMPOS_PATH = "merging/logs/tiempos_aislar_lineal.json"
def registrar(nombre, t0):
    t = json.load(open(TIEMPOS_PATH)) if os.path.exists(TIEMPOS_PATH) else {}
    t[nombre] = round(time.time() - t0, 1)
    json.dump(t, open(TIEMPOS_PATH, "w"), indent=1)
    print(f"[tiempo] {nombre}: {t[nombre] / 60:.1f} min")

## Paso 1 — Fusionar (rápido) y comprobar en el espacio de pesos

In [ ]:
G = "finetuning/checkpoints/generador{}/adapter"
ADAPTADORES = f"{G.format(1)} {G.format(2)} {G.format(3)}"
for nombre, metodo in [("peft_linear_norm", "linear"), ("peft_cat_norm", "cat")]:
    t0 = time.time()
    !python merging/fusionar_adaptadores.py --metodo {metodo} --normalizar --adaptadores {ADAPTADORES} --salida finetuning/checkpoints/{nombre} 2>&1 | tee merging/logs/fusion_{nombre}.log | tail -2
    registrar(f"fusion_{nombre}", t0)
    assert os.path.exists(f"finetuning/checkpoints/{nombre}/fusion/adapter_model.safetensors"), f"{nombre}: no se generó el adaptador"

## Paso 2 — Predicciones y métricas sobre el mismo test común

In [ ]:
os.makedirs("evaluation/reportes", exist_ok=True)
for nombre in ["peft_linear_norm", "peft_cat_norm"]:
    t0 = time.time()
    !python evaluation/generar_predicciones.py --nombre {nombre} --adapter finetuning/checkpoints/{nombre}/fusion 2>&1 | tee merging/logs/pred_{nombre}.log | tail -3
    registrar(f"predicciones_{nombre}", t0)
    assert os.path.exists(f"evaluation/predicciones/{nombre}.json"), f"{nombre}: no se generaron predicciones"
    !python evaluation/metricas_automaticas.py --predicciones evaluation/predicciones/{nombre}.json --referencias evaluation/test_comun.json --campo-prediccion traduccion_modelo --salida evaluation/reportes/{nombre}.md --salida-json evaluation/reportes/{nombre}.json | head -8
!python evaluation/tabla_comparativa.py

## Paso 3 — Empaquetar (adaptadores chicos + resultados)

In [ ]:
!rm -f resultados_aislar_lineal.zip
!zip -rq resultados_aislar_lineal.zip merging/logs finetuning/checkpoints/peft_linear_norm finetuning/checkpoints/peft_cat_norm evaluation/predicciones/peft_linear_norm.json evaluation/predicciones/peft_cat_norm.json evaluation/reportes/peft_linear_norm.json evaluation/reportes/peft_linear_norm.md evaluation/reportes/peft_cat_norm.json evaluation/reportes/peft_cat_norm.md
!ls -lh resultados_aislar_lineal.zip
from google.colab import files
files.download("resultados_aislar_lineal.zip")